## imports

In [2]:
import pandas as pd

The project CSV files are stored in the data/raw

In [3]:
ad_revenue = pd.read_csv('../data/raw/ad_revenue.csv')
iap_products = pd.read_csv('../data/raw/iap_products.csv')
matches = pd.read_csv('../data/raw/matches.csv')
purchases = pd.read_csv('../data/raw/purchases.csv')
sessions = pd.read_csv('../data/raw/sessions.csv')
ua_spend = pd.read_csv('../data/raw/ua_spend.csv')
users = pd.read_csv('../data/raw/users.csv')

In [4]:
# Date parsing helper for mixed formats
def parse_mixed_date(series):
    s = series.astype(str).str.strip()
    parsed = pd.Series(index=s.index, dtype='datetime64[ns]')
    iso_mask = s.str.fullmatch(r'\d{4}-\d{2}-\d{2}', na=False)
    slash_mask = s.str.fullmatch(r'\d{1,2}/\d{1,2}/\d{4}', na=False)
    text_mask = s.str.lower().str.fullmatch(r'\d{1,2}\s+[a-záéíóúñ]{3}\s+\d{4}', na=False)
    month_map = {'ene': '01', 'feb': '02', 'mar': '03', 'abr': '04', 'may': '05', 'jun': '06', 'jul': '07', 'ago': '08', 'sep': '09', 'oct': '10', 'nov': '11', 'dic': '12'}
    parsed.loc[iso_mask] = pd.to_datetime(s.loc[iso_mask], format='%Y-%m-%d', errors='coerce')
    parsed.loc[slash_mask] = pd.to_datetime(s.loc[slash_mask], format='%d/%m/%Y', errors='coerce')
    if text_mask.any():
        text_parts = s.loc[text_mask].str.lower().str.replace('.', '', regex=False).str.extract(r'(?P<day>\d{1,2})\s+(?P<mon>[a-záéíóúñ]{3})\s+(?P<year>\d{4})')
        normalized_text = text_parts['day'].str.zfill(2) + '/' + text_parts['mon'].map(month_map) + '/' + text_parts['year']
        parsed.loc[text_mask] = pd.to_datetime(normalized_text, format='%d/%m/%Y', errors='coerce')
    remaining = ~(iso_mask | slash_mask | text_mask)
    if remaining.any():
        parsed.loc[remaining] = pd.to_datetime(s.loc[remaining], dayfirst=True, errors='coerce')
    return parsed


file: ad_revenue

In [5]:
# Date Format
ad_revenue['fecha'] = parse_mixed_date(ad_revenue['fecha'])

#text normalize
map_type = {'recompensado':'rewarded'}
ad_revenue['tipo_ad'] = ad_revenue['tipo_ad'].str.strip().str.lower().replace(map_type)

ad_revenue['red_ad'] = ad_revenue['red_ad'].str.strip().str.lower()

file: iap_products

In [6]:
# format
iap_products['precio_base'] = iap_products['precio_base'].str.strip().str.upper().str.replace('€','', regex=False).str.replace(',','.', regex=False).astype(float)


#text normalize
iap_products['tipo'] = iap_products['tipo'].str.strip().str.lower()


file: matches

In [7]:
# Date Format
matches['fecha'] = parse_mixed_date(matches['fecha'])

# Text normalize
map_resultado = {'fail': 'lose', 'ganada':'win', 'perdida':'lose'}
matches['resultado'] = matches['resultado'].str.strip().str.lower().replace(map_resultado)

file: purchases

In [8]:
# Date Format
purchases['fecha'] = parse_mixed_date(purchases['fecha'])

def norm_plat(x):
    s = str(x).strip().lower()
    if s == 'ios':
        return 'ios'
    if s == 'and':
        return 'android'
    return s

# clearing of securities
purchases['moneda'] =purchases['moneda'].astype(str).str.strip().str.upper().replace({'€': 'EUR'})
purchases['plataforma'] = purchases['plataforma'].str.strip().str.lower().apply(norm_plat)
purchases['importe'] = purchases['importe'].str.replace('€', '', regex=False).str.replace('MXN', '', regex=False).str.replace(',', '.', regex=False).str.strip().astype(float)

sales = purchases[(purchases['reembolso'] == 0)&(purchases['moneda'] == 'EUR')].copy()

print(f"Ventas EUR validas: {len(sales)}")
print(f"Ingreso IAP bruto: {sales['importe'].sum():.2f} EUR")

Ventas EUR validas: 4479
Ingreso IAP bruto: 10838.71 EUR


file: sessions

In [9]:
# drop duplicates
sessions = sessions.drop_duplicates('evento_id')

# Date Format
sessions['fecha'] = parse_mixed_date(sessions['fecha'])

# Text Normalize
def norm_plat(x):
    s = str(x).strip().lower()
    if s == 'ios':
        return 'ios'
    if s == 'and':
        return 'android'
    return s

sessions['plataforma'] = sessions['plataforma'].str.strip().str.lower().apply(norm_plat)

valid_sessions = sessions[(sessions['duracion_segundos']>0) & (sessions['duracion_segundos']<= 3600)]

print(f"Duracion media SIN limpiar: {sessions['duracion_segundos'].mean():.1f} s")
print(f"Duracion media limpia: {valid_sessions['duracion_segundos'].mean():.1f} s")

Duracion media SIN limpiar: 586.7 s
Duracion media limpia: 183.6 s


file: ua_spend

In [10]:
# Date Format
ua_spend['fecha'] = parse_mixed_date(ua_spend['fecha'])

channel_map = {
    'coinrush': 'coinrush', 'coin_rush': 'coinrush',
    'meta': 'meta', 'facebook': 'meta',
    'google': 'google', 'google ads': 'google', 'google_ads': 'google',
    'apple_search': 'apple_search', 'asa': 'apple_search', 'apple search': 'apple_search',
    'tiktok': 'tiktok', 'tik_tok': 'tiktok',
    'organico': 'organico', 'orgánico': 'organico', 'organic': 'organico',
}

# clearing of securities
ua_spend['gasto'] = (ua_spend['gasto'].str.replace('€','', regex=False).str.replace('.','', regex=False).str.replace(',','.', regex=False).str.strip().astype(float))
ua_spend['canal'] = (ua_spend['canal'].str.strip().str.lower().map(channel_map))


print(f'Gráficas de canal en gasto: {ua_spend['canal'].nunique()}')

sumary =ua_spend.groupby('canal')['gasto'].sum().sort_values(ascending=False)
for canal, g in sumary.items():
    print(f"  {canal}: {g:.2f} EUR")

Gráficas de canal en gasto: 6
  meta: 2880.01 EUR
  google: 2100.01 EUR
  apple_search: 1980.02 EUR
  coinrush: 1259.99 EUR
  tiktok: 839.98 EUR
  organico: 0.00 EUR


file: users

In [11]:
# Date Format
users['fecha_instalacion'] = parse_mixed_date(users['fecha_instalacion'])

# clearing of securities
chanel_mapa_user = {
    'coinrush': 'coinrush', 'coin_rush': 'coinrush',
    'meta': 'meta', 'facebook': 'meta',
    'google': 'google', 'google ads': 'google', 'google_ads': 'google',
    'apple_search': 'apple_search', 'asa': 'apple_search', 'apple search': 'apple_search',
    'tiktok': 'tiktok', 'tik_tok': 'tiktok',
    'organico': 'organico', 'orgánico': 'organico', 'organic': 'organico',
}

country_map = {
    'es': 'españa', 'españa': 'españa', 
    'mx': 'méxico', 'méxico': 'méxico', 'mexico': 'méxico',  
    'pt': 'portugal',  'portugal':  'portugal', 
    'it': 'italia', 'italia':'italia'
}

# Text Normalize
def norm_plat(x):
    s = str(x).strip().lower()
    if s == 'ios':
        return 'ios'
    if s == 'and':
        return 'android'
    return s

users['canal_adquisicion'] = users['canal_adquisicion'].str.strip().str.lower().map(chanel_mapa_user) 
users['pais'] = users['pais'].str.strip().str.lower().map(country_map)
users['plataforma'] = users['plataforma'].str.strip().str.lower().apply(norm_plat)


# Export data

In [12]:
ad_revenue.to_csv('../data/processed/ad_revenue.csv', index=False)
iap_products.to_csv('../data/processed/iap_products.csv', index=False)
matches.to_csv('../data/processed/matches.csv', index=False)
sales.to_csv('../data/processed/purchases.csv', index=False)
valid_sessions.to_csv('../data/processed/sessions.csv', index=False)
ua_spend.to_csv('../data/processed/ua_spend.csv', index=False)
users.to_csv('../data/processed/users.csv', index=False)
